# Complete RR-HRV architecture - Windows walkthrough

This notebook runs the v0.2 architecture one stage at a time. It produces measurements and reliability context, not artifact or seizure labels. The requested NeuroKit 250-ms experiment is always shown beside the 300-ms software baseline.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
FEATURE_ROOT = next(path for path in candidates if (path / 'src' / 'ecg_cascade').is_dir())
PROJECT_ROOT = FEATURE_ROOT.parent
sys.path.insert(0, str(FEATURE_ROOT / 'src'))
print('Feature package:', FEATURE_ROOT)
print('Project root:', PROJECT_ROOT)

## 1. Select a WFDB record

Record 113 exposes T-wave double detections. Record 207 exposes morphology and polarity dependence. The path below points to the installed MIT-BIH database.

In [ ]:
MITDB = PROJECT_ROOT / 'Datasets' / 'mit-bih-arrhythmia-database-1.0' / 'mit-bih-arrhythmia-database-1.0.0'
RECORD = '113'
RECORD_PATH = MITDB / RECORD  # no .hea extension
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'rr_hrv_architecture' / f'notebook_record_{RECORD}'
assert RECORD_PATH.with_suffix('.hea').is_file(), RECORD_PATH
RECORD_PATH

## 2. Freeze the configuration

The selected output uses the original orientation. The inverted signal is still processed as non-routing context. The current Ho support tolerance is 50 ms; 150 ms remains an ablation.

In [ ]:
from ecg_cascade import RRHRVConfig

config = RRHRVConfig(
    processing_orientation='original',
    compute_inverted_context=True,
    neurokit_experimental_min_delay_ms=250.0,
    min_delay_inclusive=True,
    support_tolerance_ms=50.0,
    legacy_support_tolerance_ms=150.0,
    hrv_window_rr_intervals=100,
)
config.to_dict()

## 3. Run all detector and reliability paths

UNSW owns the primary timestamps. NeuroKit 250 is the experimental secondary, NeuroKit 300 is the baseline, and Pan-Tompkins is context only. This cell can take several seconds for a full 30-minute MIT-BIH record.

In [ ]:
from ecg_cascade import run_rr_hrv_wfdb

segment, result = run_rr_hrv_wfdb(
    RECORD_PATH,
    channel=0,
    config=config,
)
result.polarity_context

## 4. Inspect per-QRS support

`qrs_supported` means exactly one NeuroKit-250 event was found within +/-50 ms of the UNSW event. It is not an artifact label.

In [ ]:
event_columns = [
    'primary_candidate_time_s', 'r_fiducial_time_s',
    'secondary_match_count', 'secondary_offset_ms',
    'qrs_supported', 'qrs_supported_nk300_baseline',
    'qrs_supported_support150_ablation',
]
result.selected.primary_events[event_columns].head(12)

## 5. Inspect RR and 100-RR HRV output

Unsupported RR intervals remain present. `feature_defined` concerns only finite mathematics; `feature_reliable` also requires the configured detector-support coverage.

In [ ]:
features = result.selected.features
feature_columns = [
    'end_time_s', 'rr_ms', 'heart_rate_bpm', 'rr_supported',
    'rr_supported_nk300_baseline', 'rr_supported_support150_ablation',
    'rr_reliability_coverage', 'csi100', 'modcsi100_filtered_ms',
    'slope100_bpm_per_s', 'j1_csi_x_slope',
    'j2_modcsi_filtered_x_slope', 'feature_defined', 'feature_reliable',
]
features.loc[features['feature_defined'], feature_columns].head(10)

## 6. Compare against expert MIT-BIH beats

This validation is possible because MIT-BIH has `.atr` beat annotations. Seizure EDFs generally do not provide equivalent R-wave ground truth.

In [ ]:
from ecg_cascade.validation import evaluate_branch_against_reference
from ecg_cascade.wfdb_io import load_wfdb_beat_annotations

reference, symbols = load_wfdb_beat_annotations(RECORD_PATH)
detector_metrics, rr_metrics = evaluate_branch_against_reference(
    result, reference, sampling_rate_hz=segment.sampling_rate_hz, tolerance_ms=75.0
)
display(detector_metrics)
rr_metrics

## 7. Save the complete audit package and plot

In [ ]:
from ecg_cascade.outputs import save_rr_hrv_outputs
from ecg_cascade.plotting import save_complete_rr_hrv_diagnostic

paths = save_rr_hrv_outputs(result, OUTPUT_DIR)
plot_path = OUTPUT_DIR / 'rr_hrv_complete_diagnostic.png'
save_complete_rr_hrv_diagnostic(
    plot_path, ecg=segment.samples, result=result,
    title=f'MIT-BIH {RECORD} | {segment.channel_name}',
)
paths, plot_path

## 8. Critical interpretation

For record 113, expect UNSW to be close to the expert annotations while NeuroKit produces many T-wave double detections. Consequently, two-detector RR coverage can be low even when the primary RR sequence is accurate. For record 207, rerun with `RECORD='207'`, then compare original and inverted rows in `polarity_context`; do not automatically route solely because one row has higher agreement.